In [ ]:
%%capture
import os
import pandas as pd
from dj_notebook import activate
from pathlib import Path
env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
from intecomm_analytics.dataframes import get_appt_df
from edc_analytics.stata.get_stata_labels_from_model import strip_html
from django.apps import apps as django_apps
from intecomm_analytics.dataframes.main_1858_to_stata import df_main_variable_labels

In [ ]:
df_appt = get_appt_df()

In [ ]:
def get_stata_labels_from_model(model: str, suffix: str|None=None) -> dict[str:str]:
    """Generate STATA labels"""
    labels = {}
    _, model_name = model.split(".")
    model_cls = django_apps.get_model(model)
    for fld in model_cls._meta.get_fields():
        if suffix:
            labels.update({f"{fld.name}_{suffix}": strip_html(str(fld.verbose_name)[:80])})
        else:
            try:
                labels.update({fld.name: strip_html(str(fld.verbose_name)[:80])})
            except AttributeError:
                pass
    return labels

variable_labels = {}
variable_labels.update(**get_stata_labels_from_model("edc_appointment.appointment"))
variable_labels.update(**df_main_variable_labels())

variable_labels.update({
    "location_direction": "a->b means comm subject visited facility",
    "location_comment": "reason why comm subject visited facilty",
    "reason_missed_detail": "reason(s) why appointment was missed, missed_reasons+missed_reasons_other",
    "reason_unscheduled": "reason why appointment/visit was unscheduled, is reason+reason_other",
})


In [ ]:
df_appt = df_appt.drop(columns=["appt_type_other", "document_status_comments"])
df_appt["timepoint"] = df_appt["timepoint"].astype("Int64")
df_appt["subject_visit_id"] = df_appt["subject_visit_id"].astype("str")
df_appt["appointment_id"] = df_appt["appointment_id"].astype("str")

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d%H%M")

df_appt.to_stata(
    path=analysis_folder / f"df_followup_{timestamp}.dta",
    variable_labels=variable_labels,
    version=118,
    write_index=False,
)